# Skills Marketplace

This notebook demonstrates discovering and using a third-party skill from the [Skills Marketplace](https://skillsmp.com) — a community-curated catalog of reusable agent skills.

We will:
1. **Install** the `slack-gif-creator` skill from its [GitHub source](https://github.com/anthropics/skills/tree/main/skills/slack-gif-creator) into `.agents/skills/`
2. **Run** an `LLMAgent` that activates the skill to generate an animated GIF

The skill bundles PIL animation utilities and Slack-specific knowledge for creating optimised GIFs. It is sourced from Anthropic's public skills repository and illustrates the core value of a marketplace: a skill authored once by anyone is instantly usable by any agent, anywhere.

> **Note:** Run this notebook from within `more-examples/ch06/` so the installed skill is discovered automatically as a project-scoped skill.

## Setup Instructions

To ensure you have the required dependencies to run this notebook, you'll need to have our `llm-agents-from-scratch` framework installed on the running Jupyter kernel. To do this, you can launch this notebook with the following command while within the project's root directory:

```sh
uv run --with jupyter jupyter lab
```

Alternatively, if you just want to use the published version of `llm-agents-from-scratch` without local development, you can install it from PyPI by uncommenting the cell below.

In [ ]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPI
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have Ollama installed on your local machine and have its LLM hosting service running. To download Ollama, follow the instructions found on this page: https://ollama.com/download. After downloading and installing Ollama, you can start a service by opening a terminal and running the command `ollama serve`.

## Skill Dependencies

The `slack-gif-creator` skill uses Python's PIL library and a few imaging helpers to generate animated GIFs. Install them into the running Jupyter kernel before executing the agent cells.

In [1]:
!uv pip install pillow imageio imageio-ffmpeg numpy

Resolved 4 packages in 129ms                                         
Prepared 3 packages in 1.63s                                             
Installed 4 packages in 21ms                                
 + imageio==2.37.3
 + imageio-ffmpeg==0.6.0
 + numpy==2.4.4
 + pillow==12.2.0


## The Skill

The `slack-gif-creator` skill is [listed on the Skills Marketplace](https://skillsmp.com/skills/berriai-litellm-tests-llm-translation-test-skills-data-slack-gif-creator-skill-md). It originates from [Anthropic's skills repository](https://github.com/anthropics/skills/tree/main/skills/slack-gif-creator) and has been widely forked across the community. We will install whichever copy is currently ranked highest by stars.

The skill bundles:
- `SKILL.md` — instructions and animation knowledge for the agent
- `core/` — Python modules (`GIFBuilder`, easing functions, validators, frame helpers)

### Searching the Marketplace

The SkillsMP REST API lets you search the catalog programmatically. The query below shows how widely a single skill can spread across the community — dozens of repos have forked or mirrored `slack-gif-creator`.

In [7]:
import json
import urllib.request
from urllib.parse import urlparse

SKILLSMP_API = "https://skillsmp.com/api/v1"

with urllib.request.urlopen(
    f"{SKILLSMP_API}/skills/search?q=slack-gif-creator&limit=5&sortBy=stars",
) as resp:
    data = json.loads(resp.read())

total = data["data"]["pagination"]["total"]
print(f"Found {total} results for 'slack-gif-creator'. Top 5:\n")
for s in data["data"]["skills"]:
    print(f"  {s['name']} ({s['stars']} ⭐️) — {s['githubUrl']}")

Found 1000 results for 'slack-gif-creator'. Top 5:

  slack-gif-creator (44528 ⭐️) — https://github.com/BerriAI/litellm/tree/litellm_internal_staging/tests/llm_translation/test_skills_data/slack-gif-creator
  slack-gif-creator (34867 ⭐️) — https://github.com/sickn33/antigravity-awesome-skills/tree/main/skills/slack-gif-creator
  slack-gif-creator (2796 ⭐️) — https://github.com/davepoon/buildwithclaude/tree/main/plugins/all-skills/skills/slack-gif-creator
  slack-gif-creator (2236 ⭐️) — https://github.com/snyk/agent-scan/tree/main/tests/skills/slack-gif-creator
  slack-gif-creator (427 ⭐️) — https://github.com/Dokhacgiakhoa/antigravity-ide/tree/main/.agent/skills/slack-gif-creator


In [8]:
# Use the top-starred result as the install source
top_skill = data["data"]["skills"][0]
parts = urlparse(top_skill["githubUrl"]).path.lstrip("/").split("/")
# path format: {owner}/{repo}/tree/{branch}/{skill/path...}
REPO = f"{parts[0]}/{parts[1]}"
BRANCH = parts[3]
SKILL_PATH = "/".join(parts[4:])
print(f"\nInstalling from: {top_skill['githubUrl']}")


Installing from: https://github.com/BerriAI/litellm/tree/litellm_internal_staging/tests/llm_translation/test_skills_data/slack-gif-creator


## Installing the Skill

We use the GitHub contents API to fetch only the skill's subdirectory i.e. no need to download the entire repository. The files will land in `.agents/skills/slack-gif-creator/`, which `LLMAgent` discovers automatically as a project-scoped skill.

In [9]:
from pathlib import Path

skill_dir = Path(".agents/skills/slack-gif-creator")
skill_dir.mkdir(parents=True, exist_ok=True)


def _install(api_path: str, local_dest: Path) -> None:
    url = (
        f"https://api.github.com/repos/{REPO}/contents/{api_path}?ref={BRANCH}"
    )
    with urllib.request.urlopen(url) as resp:
        items = json.loads(resp.read())
    for item in items:
        dest = local_dest / item["name"]
        if item["type"] == "file":
            with urllib.request.urlopen(item["download_url"]) as resp:
                dest.write_bytes(resp.read())
            print(f"  {dest.relative_to(skill_dir)}")
        elif item["type"] == "dir":
            dest.mkdir(exist_ok=True)
            _install(item["path"], dest)


_install(SKILL_PATH, skill_dir)
print(f"\nInstalled to: {skill_dir.resolve()}")

  SKILL.md
  core/__init__.py
  core/easing.py
  core/frame_composer.py
  core/gif_builder.py
  core/validators.py
  requirements.txt

Installed to: /home/nerdai/Projects/llm-agents-from-scratch/more-examples/ch06/.agents/skills/slack-gif-creator


## Inspecting the Skill

Before running the agent, let's confirm the skill was discovered and take a look at its `SKILL.md`.

In [5]:
import logging

from llm_agents_from_scratch import TOOLS_FOR_SKILL_RESOURCES, LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.logger import enable_console_logging

enable_console_logging(logging.INFO)

llm = OllamaLLM(model="qwen3:14b", think=False)
agent = LLMAgent(llm=llm, tools=TOOLS_FOR_SKILL_RESOURCES)

task = Task(instruction="placeholder")
handler = LLMAgent.TaskHandler(llm_agent=agent, task=task)
skill = handler.skills["slack-gif-creator"]

print(f"Name:      {skill.frontmatter.name}")
print(f"Scope:     {skill.scope}")
print(f"Skill dir: {skill_dir.resolve()}")
print("\nResources:")
for r in skill.resources:
    print(f"  - {r}")
print()
print("--- Body ---")
print(skill.read_body()[:500])
print("--- TRUNCATED ---")

Name:      slack-gif-creator
Scope:     SkillScope.PROJECT
Skill dir: /home/nerdai/Projects/llm-agents-from-scratch/more-examples/ch06/.agents/skills/slack-gif-creator

Resources:

--- Body ---
# Slack GIF Creator

A toolkit providing utilities and knowledge for creating animated GIFs optimized for Slack.

## Slack Requirements

**Dimensions:**
- Emoji GIFs: 128x128 (recommended)
- Message GIFs: 480x480

**Parameters:**
- FPS: 10-30 (lower is smaller file size)
- Colors: 48-128 (fewer = smaller file size)
- Duration: Keep under 3 seconds for emoji GIFs

## Core Workflow

```python
from core.gif_builder import GIFBuilder
from PIL import Image, ImageDraw

# 1. Create builder
builder = GI
--- TRUNCATED ---


## Generating a GIF

Now we run the agent with the `slack-gif-creator` skill. The agent will:

1. Activate the skill via `UseSkillTool` to read its instructions
2. Write PIL animation code based on the prompt
3. Execute it with `PythonInterpreterTool` using `code=` and `cwd=skill_dir` so the bundled `core/` modules are importable
4. Save the result as `output.gif` inside the skill directory

In [8]:
OUTPUT_GIF = Path.cwd().resolve() / "output.gif"

result = await agent.run_with_skill(
    "slack-gif-creator",
    prompt=(
        "Create a spinning star emoji GIF (128x128 px, optimized for Slack)."
        f" Save the output to {OUTPUT_GIF}."
        " After reading the skill instructions, write the animation code and"
        " execute it immediately using the from_scratch__python_interpreter"
        " tool with your code as the 'code' parameter and"
        f" cwd='{skill_dir.resolve()}'.",
    ),
    max_steps=10,
)
print(result)

INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: This is a user-explicit skill activation. Call the from_scratch__use_skill tool with name='slack-gif-creator'. Use exactly this name ...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: This is a user-explicit skill activation. Call the from_scratch__use_skill tool with name='slack-gif-creator'. Use exactly this na...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      🛠️ Executing Tool Call: from_scratch__use_skill
INFO (llm_agents_fs.TaskHandler) :      ✅ Successful Tool Call: <skill_content name="slack-gif-creator">
# Slack GIF Creator

A toolkit providing utilities and knowledge for creating animate...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      ✅ Step Result: I need to create a spinning star emoji GIF (128x128 px, optimized for Slack) and save it to the specified location. I will use the `fro...[TRUNCATED]
INFO (llm_agents_fs.TaskHandler) :      🧠 New Step: Execute the provided code using the from_sc

## Result

In [ ]:
import base64

from IPython.display import HTML

with open(OUTPUT_GIF, "rb") as f:
    data = base64.b64encode(f.read()).decode()

HTML(f'<img src="data:image/gif;base64,{data}">')